# Initialization

In [0]:
import pyspark.sql.functions as f
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col

# Read Bronze table

In [0]:
df = spark.table('bikescatalog.bronze.crm_cust_info')
df.display()

# Silver Transformations

# Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

# Normailzation

In [0]:
df = (
    df
    .withColumn("cst_marital_status",
             f.when(f.upper(f.col("cst_marital_status")) == "S", "Single")
              .when(f.upper(f.col("cst_marital_status")) == 'M', "Married")
              .otherwise("n/a")
    )
    .withColumn("cst_gndr", 
                f.when(f.upper(f.col("cst_gndr")) == "M", "Male")
                .when(f.upper(f.col("cst_gndr")) == 'F', "Female")
                .otherwise("n/a")     
    )
)                                   

# Remove Records with missing customer id

In [0]:
df = df.filter(col("cst_id").isNotNull())

# Renaming Columns

In [0]:
RENAME_MAP = {
    "cst_id": "customer_id",
    "cst_key": "customer_number",
    "cst_firstname": "first_name",
    "cst_lastname": "last_name",
    "cst_marital_status":"marital_status",
    "cst_gndr": "gender",
    "cst_create_date":"created_date"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name,new_name)

# Sanity check of dataframe

In [0]:
df.limit(10).display()

# Writing silver table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("bikescatalog.silver.crm_customers")

# Sanity check of silver Table

In [0]:
%sql
select * from bikescatalog.silver.crm_customers limit 10;